# Stage 3 readiness — integration guide
## ISM 6562 — Final Project — MediStream Telehealth
---
**Not an implementation.** This notebook documents what Stage 3 (Kafka + Spark Structured Streaming) consumes from Stages 1 + 2 and gives copy-paste-ready snippets for the patterns Stage 3 will need (dimension lookups, threshold reuse, write-back paths). The actual streaming jobs live elsewhere.

Pair this with the **Week 10 assignment** (`Real-Time Streaming Pipeline`) for the prof's Stage 3 patterns and grading rubric.

## What's available from Stages 1 + 2

### Curated zone (`/medistream/curated/`)
- `appointments` — partitioned by `specialty`, `scheduled_month`. Use as a dimension table for enriching streamed `session_id → appointment_id` events.
- `session_quality` — partitioned by `device_type`. Historical per-session aggregates; Stage 3 will produce the per-sample stream.
- `patient_vitals` — partitioned by `measurement_type`. Stage 3 may want pre-visit vitals for context.
- `patient_feedback` — unpartitioned. Stage 3 may correlate alerts with downstream feedback.
- `physician_schedule` — unpartitioned. Useful for utilization context on alerts.

### Analytics zone (`/medistream/analytics/`)
Pre-computed tables Stage 3 alert enrichment can reference instead of recomputing on each micro-batch:
- `quality_by_device_os` — baseline quality per device/OS. Use to decide if a streamed metric is anomalous *for its device class*.
- `degraded_sessions` — historical degradation flags. Useful for backfilling a model that learns from historical patterns.
- `patient_history_scores` — per-patient engagement tier. Could prioritize alerts for high-value (loyal-tier) patients.
- `physician_quality_adjusted_volume` — physician ranking. Useful for routing alerts to high-rated physicians' sessions first.
- `no_show_breakdown`, `physician_perf`, `utilization_rates`, `patient_journey`, `no_show_features`, `session_quality_agg`, `session_quality` — additional dashboards/features.

## Suggested Kafka topology (matches brief's Stage 3 guide)

```
session-metrics    keyed by session_id   raw quality samples (Stage 3 producer publishes these)
session-alerts     keyed by session_id   degradation events emitted by the streaming consumer
```

### Sample alert envelope schema (proposed — keep consistent with degraded_sessions table)

```json
{
  "session_id":     "SESS-...",
  "appointment_id": "APPT-...",
  "physician_id":   "DR-...",
  "specialty":      "Cardiology",
  "alert_type":     "high_latency" | "packet_loss" | "low_resolution",
  "window_start":   "2025-04-01T14:32:00Z",
  "window_end":     "2025-04-01T14:34:00Z",
  "metrics": {
    "avg_latency_ms":    612.4,
    "avg_packet_loss":   6.1,
    "avg_audio_quality": 4.0
  },
  "suggested_action": "suggest switching to audio-only"
}
```

## Quality thresholds — share these constants with Stage 3

Already used in `02c-quality-by-device-os` and `02e-degraded-sessions`. Stage 3 alerts should fire on the same numbers so batch and streaming agree:

| Metric | Threshold | Source |
|---|---|---|
| `latency_ms` | `> 500` | brief Stage 3 guide |
| `packet_loss_pct` | `> 5` | brief Stage 3 guide |
| `audio_quality_score` | `< 5` | mid-scale on the 1–10 codec score |

Use a 2-minute sliding window (per the brief) to alert on sustained degradation rather than single-sample spikes.

## Snippets Stage 3 will use

### 1. Load the appointment dimension as a broadcast variable

550k rows × ~10 cols easily fits in driver memory and broadcasts cheaply — much faster than a streaming join against a big table for per-event enrichment.

In [ ]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
    .appName('MediStream-Stage3-Readiness-Demo')
    .master('spark://spark-master:7077')
    .config('spark.executor.memory', '512m')
    .config('spark.driver.memory', '512m')
    .getOrCreate())

HDFS_CURATED   = 'hdfs://namenode:9000/medistream/curated'
HDFS_ANALYTICS = 'hdfs://namenode:9000/medistream/analytics'
LOCAL_CURATED   = '/home/jovyan/data/curated'
LOCAL_ANALYTICS = '/home/jovyan/data/analytics'

try:
    spark.range(1).write.mode('overwrite').parquet(f'{HDFS_ANALYTICS}/_probe')
    CURATED, ANALYTICS = HDFS_CURATED, HDFS_ANALYTICS
except Exception:
    CURATED, ANALYTICS = LOCAL_CURATED, LOCAL_ANALYTICS

# The dim a streaming consumer wants in memory for cheap enrichment
appt_dim = (spark.read.parquet(f'{CURATED}/appointments')
    .select('appointment_id', 'patient_id', 'physician_id', 'specialty', 'visit_type'))
print(f'Appointment dim rows: {appt_dim.count():,}')
appt_dim.show(5, truncate=False)

### 2. Per-device baseline lookup

When a streamed sample arrives with `device_type='phone'`, `os='Android'`, you can look up the population baseline for that combination and decide if the sample is anomalous *relative to its peers* (not just against absolute thresholds).

In [ ]:
device_baselines = spark.read.parquet(f'{ANALYTICS}/quality_by_device_os')
print(f'Device/OS baseline rows: {device_baselines.count():,}')
device_baselines.select('device_type', 'os', 'p50_latency_ms', 'p95_latency_ms',
                       'avg_packet_loss_pct', 'quality_pass_rate_pct').show(truncate=False)

### 3. Pattern: write streaming alerts back to HDFS

Pseudocode for the eventual Spark Structured Streaming job. The output path slots into the existing `analytics` zone and matches the schema of the batch `degraded_sessions` table (so dashboards can union the two if needed).

```python
from pyspark.sql.types import StructType, StructField, StringType, DoubleType, TimestampType

metrics_schema = StructType([
    StructField('session_id',      StringType()),
    StructField('appointment_id',  StringType()),
    StructField('latency_ms',      DoubleType()),
    StructField('packet_loss_pct', DoubleType()),
    StructField('audio_quality_score', DoubleType()),
    StructField('event_time',      TimestampType()),
])

stream = (spark.readStream
    .format('kafka')
    .option('kafka.bootstrap.servers', 'kafka:9092')
    .option('subscribe', 'session-metrics')
    .load()
    .select(F.from_json(F.col('value').cast('string'), metrics_schema).alias('m'))
    .select('m.*'))

windowed = (stream
    .withWatermark('event_time', '2 minutes')
    .groupBy(F.window('event_time', '2 minutes', '30 seconds'), 'session_id')
    .agg(
        F.avg('latency_ms').alias('avg_latency_ms'),
        F.avg('packet_loss_pct').alias('avg_packet_loss_pct'),
    )
    .where((F.col('avg_latency_ms') > 500) | (F.col('avg_packet_loss_pct') > 5)))

# Enrich with appt dim, then write to alerts topic and HDFS
# (left as Stage 3 implementation)
```

## Open questions for Stage 3 (decide before implementing)

1. **Producer source.** Replay `session-quality.csv.gz` as a synthetic stream? Or generate fresh samples? Replaying is simpler and ensures alerts can be cross-checked against `degraded_sessions`.
2. **Watermark vs. session timeout.** The brief says "sliding window" — Spark's `window` is fine, but `groupBy(session_id).withWatermark()` with `applyInPandasWithState` is cleaner if you want true session-end semantics.
3. **Where do alerts land.** Just `session-alerts` Kafka topic, or also written to `/medistream/analytics/streaming_alerts/` for historical analysis? Recommend both.
4. **Kafka in docker-compose.** Currently not in `docker-compose.yml`. Stage 3 needs to add `kafka` and `zookeeper` services (or use `bitnami/kafka` in KRaft mode to skip ZK).